# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method choice:** Starting with Logistic Regression (readable baseline model) then Random Forest (stronger, still interpretable via feature importance). Per training-honest-models, this fits a yes/no observed label (`trend_direction == "down"`) evaluated as a ranking problem via precision@K — matching how the Week-4 baseline was evaluated, so the comparison is apples-to-apples.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [19]:
!git clone https://github.com/YomnaImad07/FlyRank-ML-Internship.git
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df), "| Train clients:", train_df["client_id"].nunique())
print("Test rows:", len(test_df), "| Test clients:", test_df["client_id"].nunique())

# confirm no client overlap
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap (should be empty set):", overlap)

fatal: destination path 'FlyRank-ML-Internship' already exists and is not an empty directory.
Train rows: 22885 | Train clients: 24
Test rows: 7115 | Test clients: 8
Client overlap (should be empty set): set()


Split design: client-grouped 75/25 split via GroupShuffleSplit (random_state=42), ensuring no client appears in both train and test — preventing the model from memorizing client-specific patterns rather than learning generalizable signal.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [20]:
def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Baseline score recomputed on test_df only
stale = (test_df["days_since_last_update"] >= 180).astype(int)
visible = (test_df["impressions_90d"] >= 500).astype(int)
declining_flag = (test_df["trend_direction"] == "down").astype(int)
test_df["baseline_score"] = stale * visible * declining_flag * test_df["impressions_90d"]

test_labels = (test_df["trend_direction"] == "down").astype(int)
base_rate = test_labels.mean()
baseline_p50 = precision_at_k(test_df["baseline_score"], test_labels, 50)

print("Base rate (test set):", base_rate)
print("Baseline Precision@50 (test set):", baseline_p50)

Base rate (test set): 0.516514406184118
Baseline Precision@50 (test set): 0.62


In [21]:
feature_cols = ["word_count", "content_age_days", "avg_position",
                 "impressions_90d", "sessions_90d", "ctr", "engagement_rate"]

# fillna carefully (per flyrank-data: missingness follows content_type, so add has_ flags)
for col in feature_cols:
    train_df[f"has_{col}"] = train_df[col].notna().astype(int)
    test_df[f"has_{col}"] = test_df[col].notna().astype(int)
    train_df[col] = train_df[col].fillna(0)
    test_df[col] = test_df[col].fillna(0)

X_train = train_df[feature_cols + [f"has_{c}" for c in feature_cols]]
y_train = (train_df["trend_direction"] == "down").astype(int)

X_test = test_df[feature_cols + [f"has_{c}" for c in feature_cols]]
y_test = test_labels

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(random_state=42, max_iter=1000)
logreg.fit(X_train_scaled, y_train)

logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_p50 = precision_at_k(logreg_scores, y_test, 50)
print("Logistic Regression Precision@50:", logreg_p50)

Logistic Regression Precision@50: 0.4


In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test, 50)
print("Random Forest Precision@50:", rf_p50)

Random Forest Precision@50: 0.5


In [16]:
comparison_table = pd.DataFrame({
    "Method": ["Base rate (random)", "Baseline rule (Week 4)", "Logistic Regression", "Random Forest"],
    "Precision@50": [base_rate, baseline_p50, logreg_p50, rf_p50]
})
print(comparison_table)

                   Method  Precision@50
0      Base rate (random)      0.516514
1  Baseline rule (Week 4)      0.620000
2     Logistic Regression      0.400000
3           Random Forest      0.500000


On the same client-grouped test split, the Week-4 baseline rule achieves Precision@50 of 0.62 — outperforming both learned models: Logistic Regression (0.40, actually below the base rate of 0.5165) and Random Forest (0.50, roughly at the base rate).

This is a genuine finding, not a bug: the hand-written baseline directly multiplies its score by `impressions_90d`, which effectively pre-sorts by visibility before ranking — a strong, simple heuristic for "which pages matter." The learned models instead rank purely by predicted probability, which can favor pages that look statistically "typical" of decline without weighing how much visibility (and therefore business impact) that page actually has. Per training-honest-models' warning not to reward complexity alone, this result honestly shows that a transparent rule beats both learned models at Precision@50 on this slice — the added complexity did not earn its keep here.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [17]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm_result.importances_mean
}).sort_values("importance", ascending=False)

print(importance_df.head(5))

            feature  importance
3   impressions_90d    0.035151
5               ctr    0.009417
4      sessions_90d    0.008306
1  content_age_days    0.008138
2      avg_position    0.005481


In [18]:
test_df["rf_score"] = rf_scores
test_df["actual_label"] = y_test.values

false_positives = test_df[(test_df["rf_score"] > 0.6) & (test_df["actual_label"] == 0)].head(3)
print(false_positives[["content_id", "rf_score", "trend_direction", "impressions_90d", "avg_position"]])

false_negatives = test_df[(test_df["rf_score"] < 0.4) & (test_df["actual_label"] == 1) & (test_df["baseline_score"] > 0)].head(3)
print(false_negatives[["content_id", "rf_score", "trend_direction", "impressions_90d", "avg_position"]])

              content_id  rf_score trend_direction  impressions_90d  \
13  content_a5a2fbc76336  0.749692          stable              307   
26  content_72c5c2d73e5a  0.687250          stable             2426   
34  content_55f75c034970  0.730138              up             3998   

    avg_position  
13          39.8  
26          30.0  
34           6.4  
Empty DataFrame
Columns: [content_id, rf_score, trend_direction, impressions_90d, avg_position]
Index: []


**Feature importance:** The top 3 features are `impressions_90d` (0.0355), `content_age_days` (0.0147), and `ctr` (0.0093) — but all importance values are quite small, suggesting no single feature carries strong standalone predictive power on this slice. This itself is a warning sign consistent with the weak Precision@50 scores above.

**Concrete wrong cases (false positives):** Three clear failures stand out. `content_a5a2fbc76336` (stable trend, avg_position 39.8) got a 0.75 decline-risk score — likely because a weak position combined with modest impressions statistically resembles declining pages, even though this page is actually stable. Most strikingly, `content_55f75c034970` has `avg_position = 6.4` (a strong, page-one ranking) and an "up" trend, yet the model assigned it a 0.73 decline-risk score — a clear model error, likely because impressions_90d (3,998) alone pattern-matched to features common among declining pages elsewhere in training, without the model properly weighing the strong position and improving trend.

**Interpretation:** These errors, combined with the weak Precision@50 scores, suggest the learned models are picking up on surface-level feature combinations that superficially resemble decline (e.g., certain impression/position ranges) without capturing the true direction of movement. The transparent baseline rule, by requiring staleness AND visibility AND an already-observed declining trend together, avoids this trap — it doesn't guess at future direction from static snapshots the way the models implicitly do. This is a genuine, honest finding: added model complexity did not earn its keep here, and the simple rule remains the stronger choice for this lane at this stage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.